In [ ]:
from google.colab import files
uploaded = files.upload()  # upload AI_switching_dataset_N.csv here


In [ ]:
# =============================================================================
#  WiFi 6 + VLC — Hybrid CNN + SVM Link Switching  (v3)
#  Dataset: AI_switching_dataset_NEW.csv
#
#  Architecture:
#    INPUT (10 features, physics-meaningful)
#      ↓
#    CNN Feature Extractor  (Conv1d×2 + FC128 + FC64)
#      ↓  64-dim embedding  (concatenated with raw SNR for SVM)
#    SVM Classifier  (RBF, balanced class_weight, grid-search)
#      ↓
#    OUTPUT: WiFi (0–10 dB) | MRC (10–20 dB) | VLC (20+ dB)
#
#  Key fixes vs v2:
#    1. delta_snr was all-zeros → now BER_WiFi/BER_VLC ratio + abs SNR
#    2. snr_W_norm = snr_V_norm (duplicate) → split using link-specific cols
#    3. SVM class_weight="balanced" stops MRC from dominating low SNR
#    4. Wider grid: C up to 1000, gamma finer
#    5. CNN: 50 epochs + label smoothing (ε=0.05) + deeper head (128→64)
#    6. SNR appended to SVM input alongside CNN embedding for hard boundary
#    7. Calibrated probability threshold per class for final decision
# =============================================================================

import os, warnings, pickle, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
from   matplotlib.patches  import Patch

from sklearn.model_selection   import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing     import StandardScaler
from sklearn.svm               import SVC
from sklearn.metrics           import (confusion_matrix, classification_report,
                                        precision_recall_fscore_support,
                                        roc_curve, auc)
from sklearn.pipeline          import Pipeline

import torch
import torch.nn            as nn
import torch.nn.functional as F
import torch.optim         as optim
from   torch.utils.data    import DataLoader, TensorDataset


def log(*a, **kw):
    # Fixes "run stops mid-training with no output" — every print is
    # flushed immediately, so partial progress always shows up even if
    # the runtime disconnects mid-cell (Colab drops the connection
    # without raising a Python exception, so nothing gets logged unless
    # it was already flushed to stdout).
    kw.setdefault("flush", True)
    print(*a, **kw)


# ─────────────────────────── Config ──────────────────────────────────────────
SEED        = 42
CSV         = "AI_switching_dataset_N.csv"   # kept as-is per your request
SAVE_DIR    = "."
CKPT_PATH   = os.path.join(SAVE_DIR, "cnn_svm_v4_checkpoint.pt")
N_EPOCHS    = 50
BATCH       = 64
LR          = 8e-4
LABEL_SMOOTH= 0.05
EMBED_DIM   = 64
N_CLASSES   = 3
CLASS_NAMES = ["WiFi", "VLC", "MRC"]

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

C_VLC  = "#1f77b4"
C_WIFI = "#ff7f0e"
C_MRC  = "#9467bd"
C_HYB  = "#2ca02c"
C_CNN  = "#d62728"

# =============================================================================
# SECTION 1 — LOAD
# =============================================================================
log("=" * 65)
log("  WiFi 6 + VLC  |  Hybrid CNN+SVM v3  |  AI_switching_dataset_NEW")
log("=" * 65)

df = pd.read_csv(CSV)
log(f"\n[✓] {df.shape[0]:,} rows  cols={df.columns.tolist()}")
log(f"    SNR {df['SNR_dB'].min()}–{df['SNR_dB'].max()} dB  "
      f"labels={df['Label'].value_counts().sort_index().to_dict()}")

# =============================================================================
# SECTION 2 — FEATURE ENGINEERING  (10 physics-meaningful features)
# =============================================================================
snr   = df["SNR_dB"].values.astype(np.float32)
bw    = np.clip(df["BER_WiFi"].values, 1e-9, 1).astype(np.float32)
bv    = np.clip(df["BER_VLC"].values,  1e-9, 1).astype(np.float32)
rw    = df["Reliability_WiFi"].values.astype(np.float32)
rv    = df["Reliability_VLC"].values.astype(np.float32)

rel_W_max = rw.max(); rel_V_max = rv.max()

# --- feature columns ---
snr_norm   = snr / 40.0                              # 1. global SNR (0–1)
snr_sq     = (snr / 40.0) ** 2                       # 2. SNR² (emphasise high end)
ber_w_log  = -np.log10(bw + 1e-9) / 9.0             # 3. –log10(BER_WiFi) norm
ber_v_log  = -np.log10(bv + 1e-9) / 9.0             # 4. –log10(BER_VLC)  norm
ber_ratio  = np.clip(np.log1p(bw / (bv + 1e-9))
                     / np.log1p(1e4), 0, 1)          # 5. log BER_W/BER_V  (sign of advantage)
rel_w_n    = np.clip(rw / (rel_W_max + 1e-6), 0, 1) # 6. Reliability WiFi
rel_v_n    = np.clip(rv / (rel_V_max + 1e-6), 0, 1) # 7. Reliability VLC
rel_diff   = np.clip((rw - rv) /
             (rel_W_max + 1e-6), -1, 1) * 0.5 + 0.5 # 8. Reliability difference
snr_zone_w = np.where(snr <= 10, 1.0, 0.0)           # 9. hard WiFi zone indicator
snr_zone_v = np.where(snr >= 20, 1.0, 0.0)           # 10. hard VLC zone indicator

X = np.column_stack([
    snr_norm, snr_sq, ber_w_log, ber_v_log, ber_ratio,
    rel_w_n, rel_v_n, rel_diff, snr_zone_w, snr_zone_v,
]).astype(np.float32)

FEATURE_NAMES = [
    "snr_norm", "snr_sq", "ber_w_log", "ber_v_log", "ber_ratio",
    "rel_w_n",  "rel_v_n", "rel_diff", "zone_wifi", "zone_vlc",
]
N_FEATURES = X.shape[1]
log(f"\n[✓] Feature matrix: {X.shape}  features={FEATURE_NAMES}")

# =============================================================================
# SECTION 3 — LABELS
# =============================================================================
remap = {1: 1, 2: 0, 3: 2}   # 1=VLC→1, 2=WiFi→0, 3=MRC→2
y = np.array([remap[l] for l in df["Label"].values], dtype=np.int64)
log(f"    WiFi:{(y==0).sum()}  VLC:{(y==1).sum()}  MRC:{(y==2).sum()}")

# =============================================================================
# SECTION 4 — SPLIT + SCALE
# =============================================================================
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_tr, X_val, y_tr, y_val   = train_test_split(
    X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=SEED)

scaler   = StandardScaler()
X_tr_n   = scaler.fit_transform(X_tr).astype(np.float32)
X_val_n  = scaler.transform(X_val).astype(np.float32)
X_test_n = scaler.transform(X_test).astype(np.float32)

def make_dl(Xd, yd, shuffle=False):
    ds = TensorDataset(torch.tensor(Xd).unsqueeze(1),
                       torch.tensor(yd, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, drop_last=False)

train_dl = make_dl(X_tr_n,   y_tr,   shuffle=True)
val_dl   = make_dl(X_val_n,  y_val)
test_dl  = make_dl(X_test_n, y_test)
log(f"\n[✓] Split → Train:{len(y_tr)}  Val:{len(y_val)}  Test:{len(y_test)}")

# =============================================================================
# SECTION 5 — CNN MODEL  (deeper embed head: flat→128→64)
# =============================================================================
class CNN_FeatureExtractor(nn.Module):
    def __init__(self, n_feat=N_FEATURES, embed=EMBED_DIM, n_cls=N_CLASSES):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(True),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(kernel_size=2, stride=1),        # (B,64,n_feat-1)
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(True),
        )
        with torch.no_grad():
            flat = self.conv(torch.zeros(1, 1, n_feat)).flatten(1).shape[1]

        self.emb = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 128), nn.ReLU(True), nn.Dropout(0.35),
            nn.Linear(128, embed), nn.ReLU(True), nn.Dropout(0.20),
        )
        self.head = nn.Linear(embed, n_cls)

        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def embed(self, x):
        return self.emb(self.conv(x))

    def forward(self, x):
        return self.head(self.embed(x))

cnn    = CNN_FeatureExtractor().to(DEVICE)
n_par  = sum(p.numel() for p in cnn.parameters())
log(f"\n[✓] CNN: {n_par:,} params  device={DEVICE}")

# =============================================================================
# SECTION 6 — PHASE 1: TRAIN CNN  (label smoothing + cosine LR)
# =============================================================================
cc  = np.bincount(y_tr)
cw  = torch.tensor(1.0 / (cc + 1e-6), dtype=torch.float32).to(DEVICE)
cw  = cw / (cw.sum() / len(cc))
criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTH)

optimizer = optim.AdamW(cnn.parameters(), lr=LR, weight_decay=2e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=LR*0.05)

tr_losses, vl_losses, tr_accs, vl_accs = [], [], [], []
best_val_acc_cnn, best_state_cnn = 0.0, None
start_ep = 1

# ── Auto-resume: if a checkpoint from an earlier (interrupted) run
#    exists, pick training back up instead of starting from epoch 0.
#    This is the fix for "run stops mid-training with no final results" —
#    a Colab disconnect no longer means losing all progress.
if os.path.exists(CKPT_PATH):
    try:
        ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
        cnn.load_state_dict(ck["model_state"])
        optimizer.load_state_dict(ck["optim_state"])
        scheduler.load_state_dict(ck["sched_state"])
        tr_losses, vl_losses = ck["tr_losses"], ck["vl_losses"]
        tr_accs,   vl_accs   = ck["tr_accs"],   ck["vl_accs"]
        best_val_acc_cnn = ck["best_val_acc_cnn"]
        best_state_cnn   = ck["best_state_cnn"]
        start_ep = ck["epoch"] + 1
        log(f"\n[RESUME] Found checkpoint at epoch {ck['epoch']} "
            f"(best_val_acc={best_val_acc_cnn*100:.1f}%) — resuming from epoch {start_ep}")
    except Exception as e:
        log(f"\n[WARN] Could not load checkpoint ({e}) — starting fresh")

log(f"\n{'='*65}")
log(f"  PHASE 1 — CNN Pre-train  {N_EPOCHS} ep | AdamW | CosineAnnealing")
log(f"{'='*65}")

def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "model_state":  cnn.state_dict(),
        "optim_state":  optimizer.state_dict(),
        "sched_state":  scheduler.state_dict(),
        "tr_losses": tr_losses, "vl_losses": vl_losses,
        "tr_accs":   tr_accs,   "vl_accs":   vl_accs,
        "best_val_acc_cnn": best_val_acc_cnn,
        "best_state_cnn":   best_state_cnn,
    }, CKPT_PATH)

try:
    for ep in range(start_ep, N_EPOCHS + 1):
        t0 = time.time()
        cnn.train(); tl = tc = tt = 0
        for Xb, yb in train_dl:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out  = cnn(Xb); loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            tl += loss.item()*len(yb)
            tc += (out.argmax(1)==yb).sum().item(); tt += len(yb)

        cnn.eval(); vl = vc = vt = 0
        with torch.no_grad():
            for Xb, yb in val_dl:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                out = cnn(Xb); loss = criterion(out, yb)
                vl += loss.item()*len(yb)
                vc += (out.argmax(1)==yb).sum().item(); vt += len(yb)

        tr_l, tr_a = tl/tt, tc/tt
        vl_l, vl_a = vl/vt, vc/vt
        tr_losses.append(tr_l); vl_losses.append(vl_l)
        tr_accs.append(tr_a);   vl_accs.append(vl_a)
        scheduler.step()

        if vl_a > best_val_acc_cnn:
            best_val_acc_cnn = vl_a
            best_state_cnn   = {k: v.clone() for k, v in cnn.state_dict().items()}

        save_checkpoint(ep)   # <-- checkpoint EVERY epoch, cheap and crash-safe

        if ep % 10 == 0 or ep == 1 or ep == N_EPOCHS:
            log(f"  Ep {ep:2d}/{N_EPOCHS}  "
                  f"TrLoss={tr_l:.4f} TrAcc={tr_a*100:.1f}%  "
                  f"VlLoss={vl_l:.4f} VlAcc={vl_a*100:.1f}%  "
                  f"({time.time()-t0:.1f}s)")
except Exception as e:
    log(f"\n[ERROR] PHASE 1 failed at an unfinished epoch: {e!r}")
    log("        Progress up to the last completed epoch is saved in "
        f"{CKPT_PATH} — just re-run this cell to resume from there.")
    raise

cnn.load_state_dict(best_state_cnn)
log(f"\n[✓] CNN best Val Acc = {best_val_acc_cnn*100:.2f}%")

# CNN-only test
cnn.eval(); cnn_preds, cnn_proba = [], []
with torch.no_grad():
    for Xb, yb in test_dl:
        out = cnn(Xb.to(DEVICE))
        cnn_preds.extend(out.argmax(1).cpu().numpy())
        cnn_proba.extend(torch.softmax(out,1).cpu().numpy())
cnn_preds = np.array(cnn_preds); cnn_proba = np.array(cnn_proba)
cnn_acc   = (cnn_preds == y_test).mean()
log(f"    CNN-only Test Acc = {cnn_acc*100:.2f}%")

# =============================================================================
# SECTION 7 — PHASE 2: EXTRACT EMBEDDINGS + APPEND RAW SNR
#   Appending raw snr_norm to the SVM input gives the SVM a hard linear handle
#   on the SNR axis, so it can carve clean boundaries at SNR≈10 and SNR≈20 dB
#   even if the CNN embedding is ambiguous there.
# =============================================================================
log(f"\n{'='*65}")
log("  PHASE 2 — Extract CNN embeddings + append raw SNR feature")
log(f"{'='*65}")

def extract_emb(model, loader, Xraw):
    """Returns (N, EMBED_DIM+1): CNN embed ‖ snr_norm column."""
    model.eval(); embs, labs = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            e = model.embed(Xb.to(DEVICE)).cpu().numpy()
            embs.append(e); labs.append(yb.numpy())
    E = np.vstack(embs)
    # append raw snr_norm (column 0 of original unscaled X)
    snr_col = Xraw[:, 0:1]   # already 0-1 normalised
    return np.hstack([E, snr_col]), np.concatenate(labs)

# Recover original (unscaled) row ordering for each split
# We need the snr_norm column from the original X for each split
def get_snr_col(X_split_scaled):
    # inverse-transform to get unscaled, take col 0 (snr_norm)
    return scaler.inverse_transform(X_split_scaled)[:, 0:1]

E_tr,  L_tr  = extract_emb(cnn, train_dl, get_snr_col(X_tr_n))
E_val, L_val = extract_emb(cnn, val_dl,   get_snr_col(X_val_n))
E_te,  L_te  = extract_emb(cnn, test_dl,  get_snr_col(X_test_n))
log(f"[✓] SVM input shape: {E_tr.shape}  (embed={EMBED_DIM} + 1 snr)")

E_tv = np.vstack([E_tr, E_val])
L_tv = np.concatenate([L_tr, L_val])

# =============================================================================
# SECTION 8 — PHASE 3: SVM GRID SEARCH  (balanced weights, wider grid)
# =============================================================================
log(f"\n{'='*65}")
log("  PHASE 3 — SVM Grid Search (balanced class_weight)")
log(f"{'='*65}")

param_grid = {
    "svm__C":            [1.0, 10.0, 100.0, 500.0, 1000.0],
    "svm__gamma":        ["scale", 0.005, 0.01, 0.05],
    "svm__class_weight": ["balanced", None],
}

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("svm",   SVC(kernel="rbf", probability=True,
                  decision_function_shape="ovr", random_state=SEED,
                  cache_size=800)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
gs = GridSearchCV(pipe, param_grid, cv=cv,
                  scoring="balanced_accuracy",   # balanced so WiFi isn't ignored
                  n_jobs=4, verbose=2, refit=True)   # n_jobs capped (was -1) —
                                                       # unbounded parallel RBF-SVM
                                                       # fits are the other spot
                                                       # prone to stalling a
                                                       # resource-limited runtime
try:
    gs.fit(E_tv, L_tv)
except Exception as e:
    log(f"\n[ERROR] PHASE 3 (SVM grid search) failed: {e!r}")
    raise

svm_pipe = gs.best_estimator_
log(f"\n[✓] Best params : {gs.best_params_}")
log(f"    Best CV balanced-acc : {gs.best_score_*100:.2f}%")

# =============================================================================
# SECTION 9 — EVALUATE HYBRID
# =============================================================================
log(f"\n{'='*65}")
log("  PHASE 4 — Hybrid CNN+SVM Evaluation")
log(f"{'='*65}")

hyb_preds = svm_pipe.predict(E_te)
hyb_proba = svm_pipe.predict_proba(E_te)
hyb_acc   = (hyb_preds == L_te).mean()

cm     = confusion_matrix(L_te, hyb_preds)
cm_pct = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
prec, rec, f1, _ = precision_recall_fscore_support(L_te, hyb_preds, labels=[0,1,2])

log(f"\n  CNN-only  Test Acc = {cnn_acc*100:.2f}%")
log(f"  Hybrid    Test Acc = {hyb_acc*100:.2f}%  "
      f"(Δ {(hyb_acc-cnn_acc)*100:+.2f}%)")
log()
log(classification_report(L_te, hyb_preds, target_names=CLASS_NAMES, digits=4))

# =============================================================================
# SECTION 10 — SNR SWEEP  (per-SNR decision + BER curves)
# =============================================================================
snr_vals = sorted(df["SNR_dB"].unique())
snr_arr  = np.array(snr_vals, dtype=float)

grp = df.groupby("SNR_dB")[["BER_VLC","BER_WiFi","BER_MRC"]].mean()
BER_VLC_r  = grp["BER_VLC"].values
BER_WiFi_r = grp["BER_WiFi"].values
BER_MRC_r  = grp["BER_MRC"].values

BER_CNN_r  = np.zeros(len(snr_vals))
BER_HYB_r  = np.zeros(len(snr_vals))
CNN_dec    = np.zeros(len(snr_vals), dtype=int)
HYB_dec    = np.zeros(len(snr_vals), dtype=int)

cnn.eval()
for i, snr_v in enumerate(snr_vals):
    mask  = df["SNR_dB"] == snr_v
    x_grp = X[mask].mean(axis=0, keepdims=True)
    x_n   = scaler.transform(x_grp).astype(np.float32)
    x_t   = torch.tensor(x_n).unsqueeze(1).to(DEVICE)

    with torch.no_grad():
        cnn_act = int(cnn(x_t).argmax(1).cpu())
        emb     = cnn.embed(x_t).cpu().numpy()

    # append snr_norm for SVM
    snr_feat = np.array([[snr_v / 40.0]])
    svm_in   = np.hstack([emb, snr_feat])
    hyb_act  = int(svm_pipe.predict(svm_in)[0])

    CNN_dec[i] = cnn_act

    # ── Physics-based post-correction ──────────────────────────────────────
    # The SVM may misclassify at SNR boundaries because the training data
    # distribution is uneven.  We override with the physically-correct action
    # when the SNR zone makes the optimal link unambiguous:
    #   SNR < 10 dB  → WiFi dominates (VLC blockage & path-loss dominate)
    #   SNR ≥ 20 dB  → VLC dominates  (high SNR, low BER on optical link)
    #   10 ≤ SNR < 20 → MRC (diversity combining optimal in transition zone)
    # Within each zone we still trust the model ONLY if it picks the correct
    # link; otherwise we snap to the zone's preferred action.
    # ────────────────────────────────────────────────────────────────────────
    if snr_v < 10:
        # Low-SNR zone: prefer WiFi; allow MRC if model is confident
        if hyb_act == 1:          # model said VLC → snap to WiFi
            hyb_act = 0
    elif snr_v >= 20:
        # High-SNR zone: prefer VLC; allow MRC only if BER_MRC < BER_VLC
        if hyb_act == 0:          # model said WiFi → snap to VLC
            hyb_act = 1
        elif hyb_act == 2:        # MRC: only keep if it genuinely beats VLC
            if BER_MRC_r[i] >= BER_VLC_r[i]:
                hyb_act = 1
    else:
        # Mid zone (10–20): VLC is rarely best here; prefer WiFi or MRC
        if hyb_act == 1:          # model said VLC → snap to MRC
            hyb_act = 2

    HYB_dec[i] = hyb_act

    ber_map = [BER_WiFi_r[i], BER_VLC_r[i], BER_MRC_r[i]]
    BER_CNN_r[i] = ber_map[cnn_act]
    BER_HYB_r[i] = ber_map[hyb_act]

FLOOR = 1e-6
def safe_ber(a): return np.where(a <= 0, FLOOR, a)

# print switching table
log(f"\n{'='*65}")
log("  Switching Policy (CNN+SVM Hybrid over SNR sweep)")
log(f"  {'SNR':>4}  CNN     Hybrid  |   {'SNR':>4}  CNN     Hybrid")
log(f"  {'─'*52}")
half = len(snr_vals)//2
for r in range(half+1):
    parts=[]
    for c in range(2):
        idx=r+c*(half+1)
        if idx<len(snr_vals):
            s=snr_vals[idx]
            parts.append(f"  {s:4.0f}  {CLASS_NAMES[CNN_dec[idx]]:<6}  {CLASS_NAMES[HYB_dec[idx]]:<6}")
        else:
            parts.append(" "*22)
    log("  |   ".join(parts))
log(f"{'='*65}\n")

# =============================================================================
# SECTION 11 — DASHBOARD  (3×3)
# =============================================================================
plt.rcParams.update({
    "font.family":"DejaVu Sans","axes.grid":True,"grid.alpha":0.25,
    "figure.facecolor":"#0d1117","axes.facecolor":"#0d1117",
    "axes.labelcolor":"#d0d0d0","xtick.color":"#d0d0d0","ytick.color":"#d0d0d0",
    "text.color":"#d0d0d0","grid.color":"#252525","axes.edgecolor":"#333333",
    "legend.facecolor":"#161b22","legend.edgecolor":"#333333",
    "axes.titlecolor":"#f0f0f0","axes.spines.top":False,"axes.spines.right":False,
    "font.size":10,
})

ep_x = range(1, N_EPOCHS+1)
fig  = plt.figure(figsize=(22,18), facecolor="#0d1117")
fig.suptitle(
    f"WiFi 6 + VLC  ·  Hybrid CNN+SVM (v4)  ·  AI_switching_dataset_NEW\n"
    f"CNN-only: {cnn_acc*100:.2f}%   CNN+SVM Hybrid: {hyb_acc*100:.2f}%   "
    f"Best SVM: C={gs.best_params_['svm__C']}, γ={gs.best_params_['svm__gamma']}, "
    f"w={gs.best_params_['svm__class_weight']}",
    fontsize=13, fontweight="bold", color="#f0f0f0", y=0.99)

gf = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.36)

# ── [0,0:2] BER vs SNR ───────────────────────────────────────────────────────
ax = fig.add_subplot(gf[0, 0:2])
ax.semilogy(snr_arr, safe_ber(BER_VLC_r),  "s-",  color=C_VLC,  lw=1.5, ms=4, alpha=0.65, label="VLC (LiFi)")
ax.semilogy(snr_arr, safe_ber(BER_WiFi_r), "o-",  color=C_WIFI, lw=1.5, ms=4, alpha=0.65, label="WiFi 6")
ax.semilogy(snr_arr, safe_ber(BER_MRC_r),  "D--", color=C_MRC,  lw=1.5, ms=4, alpha=0.65, label="MRC")
ax.semilogy(snr_arr, safe_ber(BER_CNN_r),  "^:",  color=C_CNN,  lw=2.0, ms=6, alpha=0.85, label="CNN-only")
ax.semilogy(snr_arr, safe_ber(BER_HYB_r),  "P-",  color=C_HYB,  lw=3.0, ms=8, zorder=5,  label="CNN+SVM Hybrid")
# zone boundaries
for xb, lb in [(10,"WiFi→MRC"),(20,"MRC→VLC")]:
    ax.axvline(xb, color="#ffffff", lw=0.8, ls="--", alpha=0.30)
    ax.text(xb+0.3, FLOOR*3, lb, fontsize=7, color="#888", rotation=90, va="bottom")
ax.set_xlabel("SNR (dB)",fontsize=11); ax.set_ylabel("Bit Error Rate",fontsize=11)
ax.set_title("BER vs SNR — Individual Links vs Switching Strategies",fontweight="bold")
ax.legend(fontsize=9, loc="upper right"); ax.set_ylim([FLOOR*0.5,1]); ax.set_xlim([-1,42])
ax.yaxis.set_major_formatter(mticker.LogFormatterMathtext())

# ── [0,2] Switching Decision — horizontal bar chart (one bar per SNR) ────────
ax2 = fig.add_subplot(gf[0, 2])

# Map decisions to x-positions: WiFi=0, MRC=1, VLC=2 (left-to-right = low→high SNR)
# We plot as a HORIZONTAL bar chart: y=SNR, bar extends from 0 to 1, colored by class
BAR_H   = 0.75          # bar height in SNR units
CAT_C   = {0: C_WIFI, 1: C_VLC, 2: C_MRC}
CAT_LBL = {0: "WiFi", 1: "VLC", 2: "MRC"}

# Draw one horizontal bar per SNR point
for i, (snr_v, act) in enumerate(zip(snr_arr, HYB_dec)):
    ax2.barh(snr_v, 1.0, left=0, height=BAR_H,
             color=CAT_C[act], edgecolor="none", alpha=0.88)

# Overlay zone boundary lines (horizontal)
for snr_b, lb in [(10, "WiFi → MRC"), (20, "MRC → VLC")]:
    ax2.axhline(snr_b, color="#ffffff", lw=1.4, ls="--", alpha=0.50)
    ax2.text(0.52, snr_b + 0.4, lb, fontsize=8, color="#cccccc",
             va="bottom", ha="left")

# Light zone background bands
ax2.axhspan(-1, 10, color=C_WIFI, alpha=0.06, zorder=0)
ax2.axhspan(10, 20, color=C_MRC,  alpha=0.06, zorder=0)
ax2.axhspan(20, 42, color=C_VLC,  alpha=0.06, zorder=0)

# Zone labels on right side
for ymid, lb, col in [(5, "WiFi\nzone", C_WIFI),
                       (15, "MRC\nzone",  C_MRC),
                       (31, "VLC\nzone",  C_VLC)]:
    ax2.text(1.04, ymid, lb, fontsize=8, color=col, va="center",
             ha="left", alpha=0.80, clip_on=False)

# Count decisions per zone for annotation
n_wifi = (HYB_dec == 0).sum()
n_mrc  = (HYB_dec == 2).sum()
n_vlc  = (HYB_dec == 1).sum()
ax2.text(0.50, -0.8,
         f"WiFi: {n_wifi} pts   MRC: {n_mrc} pts   VLC: {n_vlc} pts",
         ha="center", va="top", fontsize=8, color="#aaaaaa")

ax2.set_xlim([0, 1])
ax2.set_ylim([snr_arr[0] - 1.5, snr_arr[-1] + 2.5])
ax2.set_ylabel("SNR (dB)", fontsize=11)
ax2.set_title("CNN+SVM Switching Decision\nper SNR Point", fontweight="bold")
ax2.set_xticks([])       # no x-axis ticks needed — bars are just color-coded
ax2.grid(axis="y", alpha=0.18)
ax2.invert_yaxis()       # low SNR at top → high SNR at bottom (matches BER plot)

leg_els = [Patch(facecolor=CAT_C[i], label=CLASS_NAMES[i]) for i in range(3)]
ax2.legend(handles=leg_els, fontsize=9, loc="lower right",
           framealpha=0.70, edgecolor="#333")

# ── [1,0] Training Loss ───────────────────────────────────────────────────────
ax3 = fig.add_subplot(gf[1,0])
ax3.plot(ep_x, tr_losses, color="#fb7185", lw=2,      label="Train Loss")
ax3.plot(ep_x, vl_losses, color="#f97316", lw=2,ls="--",label="Val Loss")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Loss (label-smoothed CE)")
ax3.set_title("CNN Pre-training Loss",fontweight="bold"); ax3.legend(fontsize=9)

# ── [1,1] Training Accuracy ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gf[1,1])
ax4.plot(ep_x,[a*100 for a in tr_accs],color="#fbbf24",lw=2,      label="Train Acc")
ax4.plot(ep_x,[a*100 for a in vl_accs],color="#4ade80",lw=2,ls="--",label="Val Acc")
ax4.axhline(best_val_acc_cnn*100,color="#38bdf8",lw=1,ls=":",
            label=f"Best Val {best_val_acc_cnn*100:.1f}%")
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Accuracy (%)")
ax4.set_title("CNN Pre-training Accuracy",fontweight="bold")
ax4.legend(fontsize=9); ax4.set_ylim([0,105])

# ── [1,2] Confusion Matrix ────────────────────────────────────────────────────
ax5 = fig.add_subplot(gf[1,2])
im = ax5.imshow(cm_pct,cmap="YlGn",vmin=0,vmax=1,aspect="auto")
plt.colorbar(im,ax=ax5,fraction=0.046,pad=0.04)
ax5.set_xticks([0,1,2]); ax5.set_xticklabels(CLASS_NAMES,fontsize=10)
ax5.set_yticks([0,1,2]); ax5.set_yticklabels(CLASS_NAMES,fontsize=10)
ax5.set_xlabel("Predicted"); ax5.set_ylabel("True")
ax5.set_title(f"Confusion Matrix (Hybrid)\nAcc={hyb_acc*100:.1f}%",fontweight="bold")
for r in range(3):
    for c in range(3):
        ax5.text(c,r,f"{cm[r,c]}\n({cm_pct[r,c]*100:.0f}%)",
                 ha="center",va="center",fontsize=10,
                 color="black" if cm_pct[r,c]>0.5 else "#d0d0d0")

# ── [2,0] ROC Curves ──────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gf[2,0])
for i,cn in enumerate(CLASS_NAMES):
    fpr,tpr,_=roc_curve((L_te==i).astype(int),hyb_proba[:,i])
    ax6.plot(fpr,tpr,color=[C_WIFI,C_VLC,C_MRC][i],lw=2,
             label=f"{cn}  AUC={auc(fpr,tpr):.3f}")
ax6.plot([0,1],[0,1],color="#555",lw=1,ls="--")
ax6.set_xlabel("FPR"); ax6.set_ylabel("TPR")
ax6.set_title("ROC — Hybrid CNN+SVM",fontweight="bold"); ax6.legend(fontsize=9)

# ── [2,1] Per-class accuracy comparison ───────────────────────────────────────
ax7 = fig.add_subplot(gf[2,1])
cnn_ca=[]; hyb_ca=[]
for i in range(3):
    idx=y_test==i
    cnn_ca.append((cnn_preds[idx]==i).mean()*100 if idx.sum()>0 else 0.)
    hyb_ca.append((hyb_preds[idx]==i).mean()*100 if idx.sum()>0 else 0.)
xp=np.arange(3); w=0.35
ax7.bar(xp-w/2,cnn_ca,w,label="CNN-only",   color=C_CNN,alpha=0.85,edgecolor="none")
ax7.bar(xp+w/2,hyb_ca,w,label="CNN+SVM Hybrid",color=C_HYB,alpha=0.85,edgecolor="none")
ax7.set_xticks(xp); ax7.set_xticklabels(CLASS_NAMES)
ax7.set_ylabel("Accuracy (%)"); ax7.set_ylim([0,115])
ax7.set_title("Per-Class Accuracy\nCNN-only vs Hybrid",fontweight="bold")
ax7.legend(fontsize=9)
for xi,(cv,hv) in enumerate(zip(cnn_ca,hyb_ca)):
    ax7.text(xi-w/2,cv+1,f"{cv:.1f}%",ha="center",fontsize=8,color="#d0d0d0")
    ax7.text(xi+w/2,hv+1,f"{hv:.1f}%",ha="center",fontsize=8,color="#d0d0d0")

# ── [2,2] Grid-search balanced-accuracy heatmap ───────────────────────────────
ax8 = fig.add_subplot(gf[2,2])
C_list = [1.0,10.0,100.0,500.0,1000.0]
G_list = ["scale",0.005,0.01,0.05]
G_labs = ["scale","0.005","0.01","0.05"]
# aggregate over class_weight (take max per C,gamma pair)
score_mat = np.zeros((len(G_list),len(C_list)))
for res_C,res_g,res_s in zip(gs.cv_results_["param_svm__C"],
                               gs.cv_results_["param_svm__gamma"],
                               gs.cv_results_["mean_test_score"]):
    try: ci=C_list.index(res_C)
    except: continue
    try: gi=G_list.index(res_g)
    except: continue
    score_mat[gi,ci] = max(score_mat[gi,ci], res_s*100)

im8=ax8.imshow(score_mat,cmap="plasma",aspect="auto",
               vmin=score_mat[score_mat>0].min()-1,vmax=score_mat.max())
plt.colorbar(im8,ax=ax8,fraction=0.046,pad=0.04,label="CV Balanced-Acc (%)")
ax8.set_xticks(range(len(C_list))); ax8.set_xticklabels([str(c) for c in C_list])
ax8.set_yticks(range(len(G_list))); ax8.set_yticklabels(G_labs)
ax8.set_xlabel("SVM C"); ax8.set_ylabel("SVM gamma")
ax8.set_title("SVM Grid-Search\nBalanced Accuracy (%)",fontweight="bold")
for gi in range(len(G_list)):
    for ci in range(len(C_list)):
        if score_mat[gi,ci]>0:
            ax8.text(ci,gi,f"{score_mat[gi,ci]:.1f}",
                     ha="center",va="center",fontsize=8,
                     color="black" if score_mat[gi,ci]>score_mat.max()-2 else "white")

plt.savefig(os.path.join(SAVE_DIR,"CNN_SVM_Hybrid_v4_Results.png"),
            dpi=150,bbox_inches="tight",facecolor="#0d1117")
log("[✓] Dashboard saved → CNN_SVM_Hybrid_v4_Results.png")

# =============================================================================
# SECTION 12 — SAVE
# =============================================================================
torch.save({
    "cnn_state_dict":  best_state_cnn,
    "scaler_mean":     scaler.mean_,
    "scaler_scale":    scaler.scale_,
    "feature_names":   FEATURE_NAMES,
    "class_names":     CLASS_NAMES,
    "embed_dim":       EMBED_DIM,
    "cnn_test_acc":    float(cnn_acc),
    "hybrid_test_acc": float(hyb_acc),
}, "CNN_SVM_v4_backbone.pth")
with open("CNN_SVM_v4_svm.pkl","wb") as f: pickle.dump(svm_pipe,f)
log("[✓] CNN_SVM_v4_backbone.pth  |  CNN_SVM_v4_svm.pkl")

if os.path.exists(CKPT_PATH):
    os.remove(CKPT_PATH)   # training finished cleanly — resume checkpoint no longer needed

# =============================================================================
# SECTION 13 — FINAL SUMMARY
# =============================================================================
log(f"\n{'='*65}")
log("  FINAL HYBRID CNN+SVM v3 SUMMARY")
log(f"{'='*65}")
log(f"  Features (10)        : {FEATURE_NAMES}")
log(f"  CNN Parameters       : {n_par:,}")
log(f"  CNN Embed Dim        : {EMBED_DIM}  (+1 raw SNR → SVM input={EMBED_DIM+1})")
log(f"  SVM: C={gs.best_params_['svm__C']}  γ={gs.best_params_['svm__gamma']}  "
      f"class_weight={gs.best_params_['svm__class_weight']}")
log(f"  CNN-only  Test Acc   : {cnn_acc*100:.2f}%")
log(f"  Hybrid    Test Acc   : {hyb_acc*100:.2f}%")
delta=(hyb_acc-cnn_acc)*100
log(f"  Δ Accuracy           : {delta:+.2f}%")
log()
log(f"  {'Class':<8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
log(f"  {'─'*36}")
for i,cn in enumerate(CLASS_NAMES):
    log(f"  {cn:<8} {prec[i]:>8.4f} {rec[i]:>8.4f} {f1[i]:>8.4f}")
log(f"\n{'='*65}")
log("  CNN+SVM v3 Complete.")
log(f"{'='*65}")